In [ ]:
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from PIL import Image

torch.set_grad_enabled(False)  # avoid blowing up mem
device = "cuda"

In [ ]:
model_id = "google/paligemma2-3b-pt-896"
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map=device
).eval()
processor = PaliGemmaProcessor.from_pretrained(model_id)

In [2]:
def get_embeddings(text: str, image: Image.Image):
    """Copied from PaliGemmaForConditionalGeneration.forward()"""
    inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

    input_ids = inputs.input_ids
    pixel_values = inputs.pixel_values
    inputs_embeds = model.get_input_embeddings()(input_ids)
    special_image_mask = (input_ids == model.config.image_token_index).unsqueeze(-1)
    special_image_mask = special_image_mask.expand_as(inputs_embeds)
    image_features = model.get_image_features(pixel_values)
    inputs_embeds = inputs_embeds.masked_scatter(special_image_mask, image_features)
    return inputs_embeds

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

image_url = "https://github.com/zazamrykh/PicFinder/blob/main/images/doge.jpg?raw=true"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))
plt.axis("off")
_ = plt.imshow(image)

In [ ]:
embeddings = get_embeddings("<image> caption en", image)
embeddings.shape

In [ ]:
from transformer_lens import HookedTransformer

hooked_llm = HookedTransformer.from_pretrained_no_processing(
    "google/gemma-2-2b", torch_dtype=torch.bfloat16
)
hooked_llm = hooked_llm.to(device)

In [3]:
loaded_embeddings = torch.load("input_embeds.pt", weights_only=True)
loaded_attn_mask = torch.load("attention_mask.pt", weights_only=True)

In [ ]:
outputs = hooked_llm.generate(
    embeddings, max_new_tokens=30, do_sample=True, return_type="tokens"
)
print(outputs.shape)
processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
inputs = processor(text="<image>caption en", images=image, return_tensors="pt").to(
    model.device
)
outputs = model.generate(**inputs, max_new_tokens=30)
processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

gemma2 = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b",
    torch_dtype=torch.bfloat16,
    device_map=device,
).eval()
gemma2_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

In [21]:
def answer_dog_question(model, tokenizer):
    outputs = model(inputs_embeds=loaded_embeddings, attention_mask=loaded_attn_mask)
    last_logits = outputs.logits[:, -1, :]
    token_id = last_logits.argmax(-1)[0]
    return tokenizer.decode(token_id)

In [ ]:
print(answer_dog_question(model=model.language_model, tokenizer=processor.tokenizer))
print(answer_dog_question(model=gemma2, tokenizer=processor.tokenizer))
print(answer_dog_question(model=model.language_model, tokenizer=gemma2_tokenizer))
print(answer_dog_question(model=gemma2, tokenizer=gemma2_tokenizer))